## **變數與運算**

### **什麼是變數?**
*   程式用來儲存可變動資料的地方
*   記錄在電腦記憶體當中的某個位址，建立一個名稱來來對照

### **生活例子**
*   錢包與錢包裡的零錢
*   IG的愛心和按讚數
*   悠遊卡和剩餘的金額
*   轉蛋和轉蛋裡面的玩具



---

### **作業上繳設定**

1. **檔案 → 在雲端硬碟中儲存副本**
2. 對老師分享的 `2026檢定培訓營` 按 **「新增雲端硬碟捷徑」** 加到「我的雲端硬碟」（課堂會引導）
3. 確認老師給的是 **編輯者** 權限（否則無法上傳）
4. 填好下方 **DAY**（Day1、Day2…）與 **STUDENT_NAME**（你的姓名）
5. 依序執行 3 個灰色程式格：設定 → 掛載 Drive → **上繳工具**（應顯示 `上繳工具已載入（2026-06-15）`）
6. 每題流程：**寫程式 → 執行作答格 → 執行 `make_submit_button("APCS_2-1")` 格 → 按上繳**

上繳後檔案會出現在：

```
2026檢定培訓營 / Day1 / 王小明 / APCS_2-1.py
```

> `Day1`、姓名等子資料夾**不用事先建立**，第一次上繳會自動產生。
> 若改過答案，要**再執行一次** `make_submit_button(...)` 那格，再按上繳。


In [ ]:
# 老師分享的共用資料夾（捷徑到「我的雲端硬碟」後的路徑，通常不用改）
SUBMIT_BASE = "/content/drive/MyDrive/橘子蘋果/APCS/2026檢定培訓營"

# ===== 請填入 =====
DAY = "Day1"           # 統一格式：Day1、Day2、Day3...
STUDENT_NAME = "王小明"  # 你的姓名


In [ ]:
#@title 首次使用需授權 Google 帳號
from google.colab import drive
drive.mount("/content/drive")
print("Google Drive 已掛載")


In [ ]:
#@title 上傳相關設定
import os

import ipywidgets as widgets
from IPython.display import clear_output, display
from IPython import get_ipython

_NOTEBOOK_CACHE = None
SUBMIT_TOOL_VERSION = "2026-06-15"


def _user_ns():
    ip = get_ipython()
    return ip.user_ns if ip is not None else globals()


def _resolve_submit_folder():
    ns = _user_ns()
    base = ns.get("SUBMIT_BASE", "")
    day = ns.get("DAY", "").strip()
    name = ns.get("STUDENT_NAME", "").strip()

    if not base:
        raise ValueError("SUBMIT_BASE 未設定")
    if not os.path.isdir(base):
        raise ValueError(
            f"找不到共用資料夾：{base}\n"
            "請確認已掛載 Drive，且已將「2026檢定培訓營」捷徑加到「我的雲端硬碟」"
        )
    if not day:
        raise ValueError("請填 DAY（例如 Day1）")
    if not day.startswith("Day") or not day[3:].isdigit():
        raise ValueError("DAY 請用 Day1、Day2 格式")
    if not name:
        raise ValueError("請填 STUDENT_NAME（你的姓名）")

    return os.path.join(base, day, name)


def _submit_filename(question_id):
    return f"{question_id}.py"


def _cell_source_text(cell):
    src = cell.get("source", [])
    if isinstance(src, str):
        return src
    return "".join(src)


def _capture_notebook_now():
    """在執行 cell 當下讀取 Colab 畫面上的講義（不能在按鈕 callback 裡讀）。"""
    try:
        from google.colab import _message

        reply = _message.blocking_request("get_ipynb", request="", timeout_sec=10)
        if reply and "ipynb" in reply:
            return reply["ipynb"]
    except Exception:
        pass
    return None


def _extract_answer_code(nb, question_id):
    marker = f"#start-{question_id}"
    for cell in nb.get("cells", []):
        if cell.get("cell_type") != "code":
            continue
        src = _cell_source_text(cell)
        if marker not in src:
            continue
        lines = []
        for line in src.splitlines():
            stripped = line.strip()
            if stripped == marker or stripped == "#start":
                continue
            lines.append(line)
        code = "\n".join(lines).strip()
        return code + "\n" if code else ""
    raise ValueError(
        f"找不到作答區 {marker}。請確認開啟的是「_上繳」版講義，並在作答格寫好程式。"
    )


def make_submit_button(question_id):
    global _NOTEBOOK_CACHE

    nb = _capture_notebook_now()
    if nb is None:
        print("⚠️ 無法讀取講義。請在 Google Colab 中執行此格。")
        _NOTEBOOK_CACHE = None
    else:
        _NOTEBOOK_CACHE = nb
        try:
            _extract_answer_code(nb, question_id)
        except ValueError as exc:
            print(f"⚠️ {exc}")

    btn = widgets.Button(description="上繳", button_style="success", icon="cloud-upload")
    output = widgets.Output()

    def on_submit(_):
        with output:
            clear_output(wait=True)
            print("上繳中，請稍候…")
            try:
                if _NOTEBOOK_CACHE is None:
                    print("⚠️ 請先重新執行此格（make_submit_button），再按上繳。")
                    return

                if not os.path.isdir("/content/drive/MyDrive"):
                    print("⚠️ 請先執行上方的「掛載 Google Drive」cell")
                    return

                code = _extract_answer_code(_NOTEBOOK_CACHE, question_id)
                if not code.strip():
                    print("⚠️ 作答區是空的，請先寫程式，再重新執行此格，然後按上繳")
                    return

                target_dir = _resolve_submit_folder()
                os.makedirs(target_dir, exist_ok=True)

                filename = _submit_filename(question_id)
                filepath = os.path.join(target_dir, filename)

                with open(filepath, "w", encoding="utf-8") as f:
                    f.write(code)

                print("✅ 上繳成功！")
                print(f"檔案：{filename}")
                print(f"路徑：{filepath}")
            except Exception as exc:
                print(f"❌ 上繳失敗：{exc}")

    btn.on_click(on_submit)
    display(widgets.VBox([btn, output]))


print(f"上繳工具已載入（{SUBMIT_TOOL_VERSION}）")

### **APCS 2-1 認識變數**
#### **題目說明：**
#### 遊戲中，勇者都有體力值(HP)和魔力值(MP) 的設定。
#### 請建立一個程式將體力值設為 40，魔力值設為 20。
#### 使用 print() 輸出勇者的數值。

#### **輸出參考如下**

```python
體力= 40
魔力= 20
```

In [ ]:
#start-APCS_2-1


In [ ]:
make_submit_button("APCS_2-1")


---

### **APCS 2-2a 算數運算子**
#### **常用7大運算子**
|  | 加 | 減 | 乘 | 除 | 整除 | 餘數 | 次方 |
|---|---|---| --- |---|---| --- |---|
| 符號   | + | - | * | / | // | % | ** |

<br>

#### **題目說明：**
#### 在程式中練習使用變數和運算子計算以下結果，並輸出 a, b, c, d 變數的內容
#### 4 個變數 a, b, c, d 分別進行不同的運算
*   a = 50
*   b = 10 + 5
*   c = a * 3
*   d = c / a
*   e = 3 ** 2
*   f = 9 // 2


In [ ]:
#start-APCS_2-2a


In [ ]:
make_submit_button("APCS_2-2a")


#### 試試看，如果修改一下運算式的順序會發生什麼事?
*   b = 10 + 5
*   c = a * 3
*   d = c / a
*   a = 50

In [ ]:
#start-APCS_2-2a_練習


In [ ]:
make_submit_button("APCS_2-2a_練習")


---

### **APCS 2-2b 位元運算子**


| 運算子 | 名稱         | 規則          | 範例             | 
| --- | ---------- | ----------- | -------------- | 
| `\|`| 位元或 (OR)    | 任一為 1 即為 1     | `00011110 \| 00010100 = 00011110` |
| `&` | 位元與 (AND)  | 兩個都為 1 才為 1 | `00011110 & 00010100 = 00010100` | 
| `^` | 位元異或 (XOR) | 不同才為 1      | `00011110 ^ 00010100 = 00001010` |
| `~` | 位元反轉 (NOT) | 0→1, 1→0    | `~00011110 = 11100001`    |  
| `>>` | 位元右移 | a // 2 | `1000011` → `0100001`    |  
| `<<` | 位元左移 |  a*2 | `1000011` → `10000110`    |  

#### **題目說明1：**

In [ ]:
#@title **例題**
#region 題目
import ipywidgets as widgets
from IPython.display import display, clear_output, Markdown
import textwrap

def question():
    
    display(Markdown("#### 執行以下Python程式片段，其結果為何？"))

    code_block = textwrap.dedent("""\
    ```python
    Ans = 30 | 20 & 10
    print(Ans % 4)
    ```
    """)

    display(Markdown(code_block))


  # 建立元件（每次跑 cell 都新建，避免殘留）
    choices = widgets.RadioButtons(
        options=['A. 0', 'B. 1', 'C. 2', 'D. 3'],
        description='答案：',
        disabled=False
    )
    submit = widgets.Button(description='提交', button_style='primary')
    output = widgets.Output()

    def on_submit(_):
        with output:
            clear_output()
            val = choices.value
            if val.startswith('C'):
                print('✅ 正確！')
            else:
                print('❌ 再試試看～')

    # 綁定事件（避免重複掛，可以先移除舊的 handler）
    try:
        submit.on_click(on_submit, remove=True)  # ipywidgets 8 支援 remove 參數
    except TypeError:
        # 若版本不支援 remove，就略過，因為我們每次都新建元件
        pass
    submit.on_click(on_submit)

    display(choices, submit, output)

question()
#endregion 題目

#### **題目說明2：**

In [ ]:
#@title **例題**
#region 題目
import ipywidgets as widgets
from IPython.display import display, clear_output, Markdown
import textwrap

def question():
    
    display(Markdown("#### 執行以下Python程式片段，其結果為何？"))

    code_block = textwrap.dedent("""\
    ```python
    a, s = 67, 0
    while a > 0:
        s += a % 16
        a = a >> 1

    print(s % 4)
    ```
    """)

    display(Markdown(code_block))


  # 建立元件（每次跑 cell 都新建，避免殘留）
    choices = widgets.RadioButtons(
        options=['A. 0', 'B. 1', 'C. 2', 'D. 3'],
        description='答案：',
        disabled=False
    )
    submit = widgets.Button(description='提交', button_style='primary')
    output = widgets.Output()

    def on_submit(_):
        with output:
            clear_output()
            val = choices.value
            if val.startswith('D'):
                print('✅ 正確！')
            else:
                print('❌ 再試試看～')

    # 綁定事件（避免重複掛，可以先移除舊的 handler）
    try:
        submit.on_click(on_submit, remove=True)  # ipywidgets 8 支援 remove 參數
    except TypeError:
        # 若版本不支援 remove，就略過，因為我們每次都新建元件
        pass
    submit.on_click(on_submit)

    display(choices, submit, output)

question()
#endregion 題目

---

### **APCS 2-3 資料型態**

#### **常見的幾種資料型態**

| 型 態 | 中文 | 英文字義 | 轉型函式 |
|---|---|---| --- |
| int   | 整數 | integer | int(N) |
| float | 浮點數(小數) | floating point | float(N) |
| bool  | 布林(是非) | Boolean | bool(N) |
| str | 字串、純文字 | string | str(N) |


#### **題目說明：**
#### 輸入以下程式, 一共有 a, b, c, d 四個變數
* 變數 a, b 因為輸入數值，因此預設為整數
* 變數 c 為一個文字字串
* 變數 d 為一個浮點數

#### 請用print()印出以下結果
*   a 除以 b
*   c 除以 d


#### **解題想法：**
*   除法運算預設的回傳值資料型態為浮點數，所以 a/b 的輸出結果以浮點數方式呈現
*   透過 int() 將文字轉為整數

In [ ]:
#start-APCS_2-3

a = 8
b = 3

c = "10"
d = 5.0


In [ ]:
make_submit_button("APCS_2-3")


---

### **APCS 2-4 數字與文字的輸出**

#### **題目說明：**
#### 建立兩個字串變數 str1, str3
#### 和一個數值變數 temp

```python
str1 = "星期一的天氣晴, 氣溫"
temp = 28
str3 = "度"
```

#### 接著使用 print() 以下方式印出串接三個變數的輸出:
1. 使用 "+" 相連
2. 使用 “,” 分隔



In [ ]:
#start-APCS_2-4


In [ ]:
make_submit_button("APCS_2-4")


<details>
<summary>
    <font size='3', color='darkred'><b>細節提醒</b></font>
</summary>
    <p>
    <ul>
        <li>使用 "+" 相連是同個資料項，因此無空白分隔</li>
        <li>使用 "," 組合是三個資料項，每個資料項會自動使用空白分隔(sep=" ")</li>
    </ul>
    </p>

---

### **APCS 2-5 四則運算 (自我挑戰)**



#### **題目說明：**
#### 設計一個程式，有 a, b, c, d 四個變數如下:



#### 變數值如下所示，依照下列公式計算結果

> 公式 : a + b/0.3*c + 9*(d-8)

```python
a = 8
b = 7
c = -6
d = 11.11
```

#### 程式需要輸出如下"相同"的文字內容 (包含有無空格)


> 輸出的結果為:-104.01


#### **解題想法：**
*   首先 a, b, c, d 變數決定數值
*   接著輸入該行計算式 result = a+b/0.3*c+9*(d-8)
*   最後需要使用 print() 輸出計算結果

In [ ]:
#start-APCS_2-5


In [ ]:
make_submit_button("APCS_2-5")


---

### **APCS 2-6 除數和餘數 (自我挑戰)**

#### **題目說明：**
#### 請計算 123 除以 5, 6 or 7 的餘數，並且輸出結果

#### **解題想法：**
* 要獲得餘數則需要使用 % 的符號
* % 又稱為 mod，是算術運算子中的一種

In [ ]:
#start-APCS_2-6


In [ ]:
make_submit_button("APCS_2-6")


---

### **APCS 2-7 計算多項式 (自我挑戰)**
#### **題目說明：**
#### 請利用前面所學的程式語法，設計一個能夠計算多項式結果的程式
#### 數學多項式 X(100-X)(3X+5)
#### 輸出結果必須如下:
``` python
X= 10 , result= 31500
X= 50 , result= 387500
X= 100 , result= 0
```


#### **解題想法：**
* X(100-X)(3X+5) 如果轉換為程式計算應為 X*(100-X)*(3*X+5)
* 由於 X 有三種，分別為 10, 50, 100 因此需要代入和輸出 3 次

In [ ]:
#start-APCS_2-7


In [ ]:
make_submit_button("APCS_2-7")


---

## **重點複習**

*   變數可以儲存數值等各式資料。
*   如果有需要互相引用的變數，在使用前要先定義該變數的內容。
*   #start 是一個註解，用於維持程式結構。
*   資料型態有助於區別變數的類型。
*   Python 有 +, -, *, /, % 五種算術運算子，另外還有 **, // 兩種特殊的運算子。
*   str() 能將數值型態的資料轉換為文字。